## Программа поиска релевантных предложений

In [40]:
import re
import math
import numpy as np
from collections import Counter, defaultdict
from razdel import sentenize
import pymorphy3
import sentencepiece as spm
import os

with open("corpus1.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read().strip()

corpus_texts = [s.text for s in sentenize(corpus_text)]

In [41]:
morph = pymorphy3.MorphAnalyzer()

# пословная токенизация с использованием регулярного выражения
def tokenize_custom(text):
    words = re.findall(r'\b[а-яёa-z\-]+\b', text.lower())
    return [morph.parse(w)[0].normal_form for w in words]


def train_sentencepiece(input_file, model_prefix, model_type="bpe", vocab_size=2000):
    spm.SentencePieceTrainer.train(
        input=input_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type=model_type,
        character_coverage=1.0,
        split_digits=True,
        byte_fallback=False,
    )
    return spm.SentencePieceProcessor(model_file=model_prefix + ".model")

# токенизация при помощи Unigram и BPE
def tokenize_sp(text, sp):
    return sp.encode(text, out_type=str)

def build_vocabulary(tokenized_docs):
    vocab = []
    for doc in tokenized_docs:
        for token in doc:
            if token not in vocab:
                vocab.append(token)

    vocab.sort()
    token2id = {token: i for i, token in enumerate(vocab)}
    return vocab, token2id

def compute_tf_matrix(tokenized_docs, token2id):
    tf = np.zeros((len(tokenized_docs), len(token2id)), dtype=float)
    for i, doc in enumerate(tokenized_docs):
        for term in doc:
            if term in token2id:
                tf[i, token2id[term]] += 1
    return tf

def compute_df(tokenized_docs):
    df = defaultdict(int)
    for doc in tokenized_docs:
        for term in set(doc):
            df[term] += 1
    return df

def compute_tfidf_matrix(tf_matrix, tokenized_docs, token2id):
    N = len(tokenized_docs)
    df = compute_df(tokenized_docs)
    idf = np.zeros(len(token2id), dtype=float)
    for term, idx in token2id.items():
        idf[idx] = math.log(N / (df[term]), 10)
    return tf_matrix * idf

def cosine_similarity(a, b):
    an, bn = np.linalg.norm(a), np.linalg.norm(b)
    if an == 0 or bn == 0:
        return 0.0
    return float(np.dot(a, b) / (an * bn))

# токенизация и векторизация запроса
def vectorize_query(query, token2id, tokenizer_name, sp=None):
    vec = np.zeros(len(token2id), dtype=float)
    if tokenizer_name == "custom":
        tokens = tokenize_custom(query)
    elif tokenizer_name in {"unigram", "bpe"}:
        tokens = sp.encode(query, out_type=str)

    for t in tokens:
        if t in token2id:
            vec[token2id[t]] += 1
    return vec, tokens

# функция поиска
def find_most_relevant(query, corpus_texts, tokenized_docs, token2id, tf_matrix, tfidf_matrix, tokenizer_name, sp=None, topk=10):
    q_tf, q_tokens = vectorize_query(query, token2id, tokenizer_name, sp)
    df = compute_df(tokenized_docs)
    N = len(tokenized_docs)
    q_tfidf = np.zeros(len(token2id), dtype=float)
    for term, idx in token2id.items():
        if q_tf[idx] > 0:
            idf = math.log(N / (df.get(term, 1)))
            q_tfidf[idx] = q_tf[idx] * idf

    sims_tf = [cosine_similarity(q_tf, tf_matrix[i]) for i in range(N)]
    sims_tfidf = [cosine_similarity(q_tfidf, tfidf_matrix[i]) for i in range(N)]

    print(f"\n===== {tokenizer_name.upper()} =====")
    print("\n--- TF ---")
    count = 0
    for i, s in sorted(enumerate(sims_tf), key=lambda x: x[1], reverse=True)[:topk]:
        count += 1
        print(f"{count}. ({s:.4f}) {corpus_texts[i]}")
        
    print("\n--- TF-IDF ---")
    count = 0
    for i, s in sorted(enumerate(sims_tfidf), key=lambda x: x[1], reverse=True)[:topk]:
        count += 1
        print(f"{count}. ({s:.4f}) {corpus_texts[i]}")
        

In [42]:
# пословная токенизация
tokenized_custom = [tokenize_custom(t) for t in corpus_texts]
vocab_custom, token2id_custom = build_vocabulary(tokenized_custom)
tf_custom = compute_tf_matrix(tokenized_custom, token2id_custom)
tfidf_custom = compute_tfidf_matrix(tf_custom, tokenized_custom, token2id_custom)

# Unigram
sp_unigram = train_sentencepiece("corpus1.txt", "sp_unigram", "unigram", 2000)
tokenized_unigram = [tokenize_sp(t, sp_unigram) for t in corpus_texts]
vocab_unigram, token2id_unigram = build_vocabulary(tokenized_unigram)
tf_unigram = compute_tf_matrix(tokenized_unigram, token2id_unigram)
tfidf_unigram = compute_tfidf_matrix(tf_unigram, tokenized_unigram, token2id_unigram)

# BPE
sp_bpe = train_sentencepiece("corpus1.txt", "sp_bpe", "bpe", 2000)
tokenized_bpe = [tokenize_sp(t, sp_bpe) for t in corpus_texts]
vocab_bpe, token2id_bpe = build_vocabulary(tokenized_bpe)
tf_bpe = compute_tf_matrix(tokenized_bpe, token2id_bpe)
tfidf_bpe = compute_tfidf_matrix(tf_bpe, tokenized_bpe, token2id_bpe)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: corpus1.txt
  input_format: 
  model_prefix: sp_unigram
  model_type: UNIGRAM
  vocab_size: 2000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  different

In [43]:
test_queries = [
    "Памятник на месте рождения Пушкина стоит не там.",
    "Одна из самых известных сцен в истории Голливуда происходит на кукурузном поле.",
    "Победный гол сальвадорского футболиста (на илл.) в ворота соперника привёл к шестидневной войне."
]

In [44]:
for query in test_queries:
    print("\n\nЗапрос: ", query)
    find_most_relevant(query, corpus_texts, tokenized_custom, token2id_custom, tf_custom, tfidf_custom, "custom")
    find_most_relevant(query, corpus_texts, tokenized_unigram, token2id_unigram, tf_unigram, tfidf_unigram, "unigram", sp_unigram)
    find_most_relevant(query, corpus_texts, tokenized_bpe, token2id_bpe, tf_bpe, tfidf_bpe, "bpe", sp_bpe)
    print("\n---------------------------------------------------------------------------------------------------------")



Запрос:  Памятник на месте рождения Пушкина стоит не там.

===== CUSTOM =====

--- TF ---
1. (0.5189) Памятник Пушкину на Бауманской улице в Москве — скульптурное изображение (бюст), установленное на месте предполагаемого рождения русского поэта и писателя Александра Сергеевича Пушкина.
2. (0.4226) Более поздние исследования, проведённые уже после установки памятника, показывают, что место рождения Пушкина скорее находится на соседней Малой Почтовой улице.
3. (0.2673) Деревянный дом № 40 был снесён в советское время, на его месте была построена школа № 345, названная именем Пушкина.
4. (0.2474) Отличие памятника от многочисленных других памятников Пушкину состоит в том, что он изображает поэта в юном возрасте.
5. (0.2315) В то время Ямайка забила 4 гола (1 место), Кюрасао — 2 мяча (2 место), а Сальвадор (3 место) поразил ворота соперников лишь единожды.
6. (0.2097) В декабре 1969 года на границе между Сальвадором и Гондурасом имели место вооружённые столкновения.
7. (0.2037) == Истор

## Анализ результатов

Пронумеруем запросы:
1. Памятник на месте рождения Пушкина стоит не там.  
2. Одна из самых известных сцен в истории Голливуда происходит на кукурузном поле.  
3. Победный гол сальвадорского футболиста (на илл.) в ворота соперника привёл к шестидневной войне."

### Собственная реализация токенизатора

Можно заметить, что при использовании tf-idf первые 6 выданных предложений полностью из статей, связанных с запросом №1, при использовании же tf 5-6 выдачи из совершенно не связанных статей. Скорее всего здесь помогло свойство tf-idf выносить вперед предложения с более редкими словами, то есть учет таких слов, как "памятник", "Пушкин".

Для запроса №2 в целом не наблюдается разницы в качестве обнаружения для tf и tf-idf: обе модели смогли найти 4 релевантных предложения среди первых 10.

Для запроса №3 также не наблюдается разницы, обе модели справились хорошо - все 10 предложений из выдачи оказались релевантны и предложение, подтверждающее факт, распологается на 1 месте для обеих моделей.

## Unigram

Для запроса №1 tf-idf показала немного более хорошие результаты, чем tf (6 предложений релевантны против 5).  
То же самое наблюдается и для запроса №2, при этом при использовании tf-idf релевантные предложения больше "проталкиваются" вперед.  
Наиболее сильно наблюдается разница для запроса №3: 10/10 релевантных предложений при tf-idf против 5/10.

При этом можно наблюдать ухудшение качества выдачи по сравнению с пословной токенизацией.

## BPE

Для запроса №1 при использовании tf-idf выдалось 7/10 релевантных предложений против 4/10 при использовании tf.  
Для запроса №2 6/10 релевантных предложений против 4/10 в пользу tf.idf.  
Для запроса №3 количество релевантных запросов было одиновым для обеих моделей (5/10).


## Выводы:
При использовании любого способа токенизации почти всегда использование модели tf-idf улучшало качество выдачи, по сравнению использования tf.   
Пословная токенизация в целом справилась с задачей лучше всего, предложение с нужным фактом почти всегда было среди первых релевантных предложений, хотя можно наблюдать, что иногда лучше справляются модели Unigram (запрос №2) или BPE (запрос №1).  
Если сравнивать BPE и Unigram, лучше справился Unigram: для 3 запроса с использованием tf-idf получилось добиться выдачи 10/10. 
В целом для 3 запроса было больше релевантных предложений для всех моделей, так как в корпусе преобладает текст, связанный с этим запросом. То есть этот пример показывает, что для качества ранжирования важен объем информации, и необходимо учитывать это для запросов в областях с недостаточным количеством информации.